# 🔐 SESA — ML Model Training on Google Colab
**Solidity Explainable Static Analyzer (SESA) — Phase 2**

This notebook trains a lightweight `RandomForestClassifier` on extracted Solidity function features from the [SmartBugs Curated](https://github.com/smartbugs/smartbugs-curated) dataset.

**Workflow:**
1. Install dependencies
2. Clone SmartBugs & run Slither to extract features
3. Train the RandomForest model
4. Evaluate & export `model.pkl`
5. Download `model.pkl` → place in `ml/` folder of your project

## Step 1: Install Dependencies

In [ ]:
!pip install slither-analyzer scikit-learn pandas joblib shap matplotlib seaborn -q
!pip install solc-select -q
!solc-select install 0.8.0
!solc-select use 0.8.0
print('✅ Dependencies installed.')

## Step 2: Clone SmartBugs Curated Dataset
We use a curated subset of labeled Solidity contracts.

In [ ]:
import os

if not os.path.exists('smartbugs-curated'):
    !git clone --depth=1 https://github.com/smartbugs/smartbugs-curated.git
    print('✅ Dataset cloned.')
else:
    print('✅ Dataset already exists.')

# Inspect available vulnerability categories
categories = os.listdir('smartbugs-curated/dataset')
print(f'\nAvailable vulnerability categories: {categories}')

## Step 3: Extract Features Using Slither
We run Slither on each contract and extract the feature vector defined in `feature_extractor.py`.

In [ ]:
import json
import pandas as pd
from slither.slither import Slither
from slither.slithir.operations import HighLevelCall, LowLevelCall, Send, Transfer, Assignment

FEATURE_NAMES = [
    "num_nodes", "num_lines", "num_parameters", "num_modifiers",
    "is_payable", "is_constructor", "visibility_score", "num_external_calls",
    "num_internal_calls", "num_state_reads", "num_state_writes",
    "has_tx_origin", "has_msg_value", "can_send_eth", "has_assembly",
    "calls_before_state_update"
]

# Categories considered 'vulnerable' for labeling
VULNERABLE_CATEGORIES = [
    'reentrancy', 'access_control', 'arithmetic', 'unchecked_low_level_calls',
    'denial_of_service', 'bad_randomness', 'front_running', 'time_manipulation',
    'short_addresses', 'other'
]
SAFE_CATEGORIES = ['safe']

def extract_features(contract_path):
    """Extract features from a single Solidity file."""
    try:
        slither = Slither(contract_path, disable_color=True)
    except Exception as e:
        return []

    features_list = []
    for contract in slither.contracts:
        if contract.is_interface or contract.is_library:
            continue
        for function in contract.functions:
            if function.is_shadowed or function.full_name.startswith('slither_'):
                continue
            try:
                feat = {
                    "num_nodes": len(function.nodes),
                    "num_lines": len(function.source_mapping.lines) if function.source_mapping else 0,
                    "num_parameters": len(function.parameters),
                    "num_modifiers": len(function.modifiers),
                    "is_payable": 1 if getattr(function, 'payable', False) else 0,
                    "is_constructor": 1 if function.is_constructor else 0,
                    "visibility_score": {"public": 3, "external": 2, "internal": 1, "private": 0}.get(function.visibility, 0),
                    "num_external_calls": len(function.high_level_calls),
                    "num_internal_calls": len(function.internal_calls),
                    "num_state_reads": len(function.all_state_variables_read()),
                    "num_state_writes": len(function.all_state_variables_written()),
                }
                all_irs = [str(ir).lower() for node in function.nodes for ir in node.irs]
                feat["has_tx_origin"] = 1 if any('tx.origin' in ir for ir in all_irs) else 0
                feat["has_msg_value"] = 1 if any('msg.value' in ir for ir in all_irs) else 0
                feat["can_send_eth"] = 1 if function.can_send_eth else 0
                feat["has_assembly"] = 1 if getattr(function, 'contains_assembly', False) else 0
                state_write_after_call, found_call = 0, False
                for node in function.nodes:
                    if any(isinstance(ir, (HighLevelCall, LowLevelCall, Send, Transfer)) for ir in node.irs):
                        found_call = True
                    if found_call and any(isinstance(ir, Assignment) for ir in node.irs):
                        state_write_after_call = 1
                        break
                feat["calls_before_state_update"] = state_write_after_call
                features_list.append(feat)
            except Exception:
                continue
    return features_list

print('✅ Feature extractor defined.')

In [ ]:
import glob

all_rows = []
total_contracts = 0
failed_contracts = 0

DATASET_BASE = 'smartbugs-curated/dataset'

# Process vulnerable contracts
for category in VULNERABLE_CATEGORIES:
    category_path = os.path.join(DATASET_BASE, category)
    if not os.path.exists(category_path):
        continue
    sol_files = glob.glob(os.path.join(category_path, '**', '*.sol'), recursive=True)
    for sol_file in sol_files[:30]:  # Limit per category for speed
        total_contracts += 1
        feats = extract_features(sol_file)
        for f in feats:
            f['is_vulnerable'] = 1
            f['source_category'] = category
        all_rows.extend(feats)

# Process safe contracts
for category in SAFE_CATEGORIES:
    category_path = os.path.join(DATASET_BASE, category)
    if not os.path.exists(category_path):
        continue
    sol_files = glob.glob(os.path.join(category_path, '**', '*.sol'), recursive=True)
    for sol_file in sol_files[:50]:
        total_contracts += 1
        feats = extract_features(sol_file)
        for f in feats:
            f['is_vulnerable'] = 0
            f['source_category'] = 'safe'
        all_rows.extend(feats)

df = pd.DataFrame(all_rows)
print(f'\n✅ Extraction complete!')
print(f'   Total contracts processed: {total_contracts}')
print(f'   Total function samples: {len(df)}')
print(f'   Vulnerable functions: {df["is_vulnerable"].sum()}')
print(f'   Safe functions: {(df["is_vulnerable"] == 0).sum()}')

# Save features CSV for inspection
df.to_csv('features_dataset.csv', index=False)
print('\n📁 features_dataset.csv saved.')

## Step 4: Train the RandomForest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data
df_clean = df.dropna(subset=FEATURE_NAMES + ['is_vulnerable'])
X = df_clean[FEATURE_NAMES].fillna(0).values
y = df_clean['is_vulnerable'].values

print(f'Dataset shape: {X.shape}')
print(f'Label distribution: {dict(zip(*map(list, (lambda u,c: (u,c))(*__import__("numpy").unique(y, return_counts=True))))}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',  # Handle imbalanced dataset
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print('\n📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Safe', 'Vulnerable']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Safe', 'Vulnerable'],
            yticklabels=['Safe', 'Vulnerable'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('\n📁 confusion_matrix.png saved.')

## Step 5: Feature Importance (SHAP Analysis)

In [ ]:
import shap
import numpy as np

# Use a sample for SHAP (faster)
sample_size = min(200, len(X_test))
X_sample = X_test[:sample_size]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print('✅ SHAP values computed.')

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_sample, feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Feature Importance — Vulnerable Class')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 shap_summary.png saved.')

## Step 6: Export Model & Download
Save the model as `model.pkl` and download it to place in your `ml/` folder.

In [ ]:
joblib.dump(model, 'model.pkl')
print('✅ model.pkl saved!')
print('\n📥 Downloading model.pkl...')

try:
    from google.colab import files
    files.download('model.pkl')
    print('\n🎉 Download triggered! Place model.pkl in your project at:')
    print('   solidity-analyzer/ml/model.pkl')
except ImportError:
    print('Not running in Colab. model.pkl saved locally.')

print('\n🚀 Next step: Run your analyzer with --ml flag:')
print('   python main.py tests/Reentrancy.sol --ml')

---
## Optional: Load Pre-existing features_dataset.csv
If you already have extracted features (e.g., from a previous run), use this cell instead of running the extraction above.

In [ ]:
# OPTIONAL: Load pre-extracted features
# df = pd.read_csv('features_dataset.csv')
# print(f'Loaded {len(df)} samples from CSV.')
# print(df.head())